# Instant3DWeb2

Automatic segmentation bridge between **SegRef3D** and **TotalSegmentator**. This notebook does not use Gradio.

1. In Colab, choose **Runtime > Change runtime type > T4 GPU** (recommended).
2. Run the setup cell.
3. Upload `instant3d_request.zip` exported by SegRef3D.
4. Validate and run TotalSegmentator.
5. Run the final download cell, then import `instant3d_result.zip` into SegRef3D.

The request ZIP contains the source NIfTI volume. Confirm that Google Colab use is permitted by your institution. Segmentation output must be reviewed and refined before research or medical use.


In [ ]:
#@title 1. Setup Instant3DWeb2
!pip -q install TotalSegmentator nibabel numpy Pillow scipy
import os, sys, shutil, subprocess
REPO = '/content/SegRef3D'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SatoruMuro/SegRef3D.git', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'ColabNotebooks'))
sys.path.insert(0, os.path.join(REPO, 'SegRef3D'))
print('Setup complete. GPU runtime is recommended.')


In [ ]:
#@title 2. Upload instant3d_request.zip
from google.colab import files
from pathlib import Path
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one instant3d_request.zip file.')
REQUEST_ZIP = str(Path('/content') / zip_names[0])
Path(REQUEST_ZIP).write_bytes(uploaded[zip_names[0]])
print('Uploaded:', REQUEST_ZIP)


In [ ]:
#@title 3. Validate request
import tempfile
from instant3d_bridge import validate_request_zip
with tempfile.TemporaryDirectory() as folder:
    manifest, source_path = validate_request_zip(REQUEST_ZIP, folder)
print('Request ID:', manifest['request_id'])
print('Source:', manifest['source']['shape'], manifest['source']['voxel_spacing_mm'], manifest['source']['orientation'])
print('Objects:')
for item in manifest['objects']:
    print(f"  Obj {item['object_id']}: {item['display_name']} ({item['task']}/{item['roi']})")


In [ ]:
#@title 4. Run TotalSegmentator and build result ZIP
from instant3dweb2_backend import process_request
RESULT_ZIP = str(process_request(REQUEST_ZIP, '/content/instant3d_result.zip'))
print('Processing complete:', RESULT_ZIP)


## Result contents

The result ZIP contains geometry-preserving binary NIfTI masks, a merged labelmap NIfTI, SegRef3D-compatible label PNGs, `volumes.csv`, and a reproducibility manifest. Lower object IDs have priority only in the merged labelmap when ROIs overlap; individual binary masks preserve overlaps.


In [ ]:
#@title 5. Download instant3d_result.zip
from google.colab import files
from pathlib import Path
if not Path(RESULT_ZIP).is_file():
    raise FileNotFoundError('Run the processing cell before downloading the result.')
files.download(RESULT_ZIP)
